# NER-Enhanced ASKQE Pipeline for BioMQM

This notebook runs the full NER-Enhanced ASKQE pipeline:
1. **NER Extraction** - Extract biomedical entities using BioBERT
2. **Entity-Aware QG** - Generate entity-specific questions using Qwen
3. **QA Source** - Answer questions on source sentences
4. **QA BT per language** - Answer questions on backtranslated sentences
5. **Mapping** - Combine source and BT answers
6. **String Comparison** - Calculate F1/EM per entity type
7. **SBERT** - Calculate semantic similarity per entity type

## Environment Setup

Detect environment (Kaggle/Colab/Local) and configure paths accordingly.

In [ ]:
import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(DRIVE_CACHE_DIR, 'sentence_transformers')
    
    print(f'Model cache directory: {DRIVE_CACHE_DIR}')
elif IN_KAGGLE:
    print('Running on Kaggle - models will be cached in /root/.cache')
else:
    print('Running locally - using default cache directories')

In [ ]:
import subprocess

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 
                'transformers', 'torch', 'accelerate', 'nltk', 
                'sentence-transformers', 'sacrebleu', 'textstat'], check=True)
print('Dependencies installed!')

In [ ]:
# Clone repository based on environment
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        print(f'Cloning repository to {PROJECT_ROOT}...')
        subprocess.run(['git', 'clone', 
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', 
                        PROJECT_ROOT], check=True)
        print('Clone complete!')
    else:
        print(f'Repository already exists at {PROJECT_ROOT}')
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone', 
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', 
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results Qwen3B baseline')
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Results directory: {RESULTS_DIR}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import torch

print('=== Downloading/Loading Models ===')
print('This may take a while on first run...\n')

MODELS = {
    'qwen': 'Qwen/Qwen2.5-3B-Instruct',
    'sbert': 'sentence-transformers/all-MiniLM-L6-v2',
    'biobert_ner': 'alvaroalon2/biobert_diseases_ner'
}

# Download Qwen model
print(f"[1/3] Loading {MODELS['qwen']}...")
tokenizer = AutoTokenizer.from_pretrained(MODELS['qwen'])
model = AutoModelForCausalLM.from_pretrained(MODELS['qwen'], torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      ✓ Qwen cached')

# Download SBERT model
print(f"[2/3] Loading {MODELS['sbert']}...")
sbert_model = SentenceTransformer(MODELS['sbert'])
del sbert_model
print('      ✓ SBERT cached')

# Download BioBERT NER model
print(f"[3/3] Loading {MODELS['biobert_ner']}...")
from transformers import pipeline
ner_pipe = pipeline('ner', model=MODELS['biobert_ner'], aggregation_strategy='simple')
del ner_pipe
print('      ✓ BioBERT NER cached')

print('\n=== All models cached! ===')

## Path Configuration

All paths are derived from PROJECT_ROOT.

In [ ]:
# ============================================
# PATH CONFIGURATION - Derived from PROJECT_ROOT
# ============================================

EXTENSION_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/ner-extension"
CODE_DIR = f"{EXTENSION_DIR}/code"
ORIGINAL_DATASET_PATH = f"{PROJECT_ROOT}/biomqm/dev_with_backtranslation.jsonl"

# Output directories
NER_OUTPUT = f"{EXTENSION_DIR}/QG/ner_output.jsonl"
QG_OUTPUT = f"{EXTENSION_DIR}/QG/qg_entity_aware.jsonl"
QA_SOURCE_OUTPUT = f"{EXTENSION_DIR}/QA/unique/source.jsonl"
QA_BT_DIR = f"{EXTENSION_DIR}/QA/unique"
MAPPED_OUTPUT = f"{EXTENSION_DIR}/QA/mapped/all.jsonl"
STRING_COMP_OUTPUT = f"{EXTENSION_DIR}/evaluation/string-comparison"
SBERT_OUTPUT = f"{EXTENSION_DIR}/evaluation/sbert"

LANGUAGES = ["de", "es", "fr", "ru", "zh-CN"]

# Optional: limit samples for testing (set to None for full run)
MAX_SAMPLES = None  # e.g., 10 for testing

# Create output directories
os.makedirs(f"{EXTENSION_DIR}/QG", exist_ok=True)
os.makedirs(f"{EXTENSION_DIR}/QA/unique", exist_ok=True)
os.makedirs(f"{EXTENSION_DIR}/QA/mapped", exist_ok=True)
os.makedirs(STRING_COMP_OUTPUT, exist_ok=True)
os.makedirs(SBERT_OUTPUT, exist_ok=True)

# Add code directory to path
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"EXTENSION_DIR: {EXTENSION_DIR}")
print(f"CODE_DIR: {CODE_DIR}")
print(f"ORIGINAL_DATASET_PATH: {ORIGINAL_DATASET_PATH}")

In [ ]:
# Verify critical files exist
print("\n=== Verifying Files ===")

code_files = [
    'ner_extraction.py',
    'qg_entity_aware.py', 
    'qa_entity.py',
    'mapping.py',
    'string_comparison.py',
    'sbert.py',
    'utils.py'
]

all_ok = True
for f in code_files:
    path = os.path.join(CODE_DIR, f)
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"{status} {f}")
    if not exists:
        all_ok = False

if os.path.exists(ORIGINAL_DATASET_PATH):
    with open(ORIGINAL_DATASET_PATH, 'r') as f:
        n_lines = sum(1 for _ in f)
    print(f"\n✓ Dataset found ({n_lines} lines)")
else:
    print(f"\n✗ Dataset NOT found: {ORIGINAL_DATASET_PATH}")
    all_ok = False

if all_ok:
    print("\n=== All files verified! ===")
else:
    print("\n=== WARNING: Some files missing! ===")

## Step 1: NER Extraction

Extract biomedical entities (DISEASE, DRUG, etc.) from source sentences.

In [ ]:
# NER Extraction

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/ner_extraction.py",
    "--input_path", ORIGINAL_DATASET_PATH,
    "--output_path", NER_OUTPUT
]

if MAX_SAMPLES:
    cmd.extend(["--max_samples", str(MAX_SAMPLES)])

print("Running NER extraction...")
subprocess.run(cmd, check=True)
print("✓ NER Extraction complete!")

## Step 2: Entity-Aware Question Generation

Generate entity-specific questions using Qwen.

In [ ]:
# Entity-Aware QG

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/qg_entity_aware.py",
    "--input_path", NER_OUTPUT,
    "--output_path", QG_OUTPUT
]

if MAX_SAMPLES:
    cmd.extend(["--max_samples", str(MAX_SAMPLES)])

print("Running Entity-Aware QG...")
subprocess.run(cmd, check=True)
print("✓ QG complete!")

## Step 3: QA - Source

Answer entity-specific questions on source sentences.

In [ ]:
# QA Source

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/qa_entity.py",
    "--mode", "source",
    "--input_path", QG_OUTPUT,
    "--output_path", QA_SOURCE_OUTPUT
]

if MAX_SAMPLES:
    cmd.extend(["--max_samples", str(MAX_SAMPLES)])

print("Running QA Source...")
subprocess.run(cmd, check=True)
print("✓ QA Source complete!")

## Step 4a: QA - German (de)

In [ ]:
# QA BT - German
lang = "de"

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/qa_entity.py",
    "--mode", "bt",
    "--lang", lang,
    "--input_path", QG_OUTPUT,
    "--bt_path", ORIGINAL_DATASET_PATH,
    "--output_path", f"{QA_BT_DIR}/bt-{lang}.jsonl"
]

if MAX_SAMPLES:
    cmd.extend(["--max_samples", str(MAX_SAMPLES)])

print(f"Running QA BT {lang}...")
subprocess.run(cmd, check=True)
print(f"✓ QA BT {lang} complete!")

## Step 4b: QA - Spanish (es)

In [ ]:
# QA BT - Spanish
lang = "es"

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/qa_entity.py",
    "--mode", "bt",
    "--lang", lang,
    "--input_path", QG_OUTPUT,
    "--bt_path", ORIGINAL_DATASET_PATH,
    "--output_path", f"{QA_BT_DIR}/bt-{lang}.jsonl"
]

if MAX_SAMPLES:
    cmd.extend(["--max_samples", str(MAX_SAMPLES)])

print(f"Running QA BT {lang}...")
subprocess.run(cmd, check=True)
print(f"✓ QA BT {lang} complete!")

## Step 4c: QA - French (fr)

In [ ]:
# QA BT - French
lang = "fr"

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/qa_entity.py",
    "--mode", "bt",
    "--lang", lang,
    "--input_path", QG_OUTPUT,
    "--bt_path", ORIGINAL_DATASET_PATH,
    "--output_path", f"{QA_BT_DIR}/bt-{lang}.jsonl"
]

if MAX_SAMPLES:
    cmd.extend(["--max_samples", str(MAX_SAMPLES)])

print(f"Running QA BT {lang}...")
subprocess.run(cmd, check=True)
print(f"✓ QA BT {lang} complete!")

## Step 4d: QA - Russian (ru)

In [ ]:
# QA BT - Russian
lang = "ru"

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/qa_entity.py",
    "--mode", "bt",
    "--lang", lang,
    "--input_path", QG_OUTPUT,
    "--bt_path", ORIGINAL_DATASET_PATH,
    "--output_path", f"{QA_BT_DIR}/bt-{lang}.jsonl"
]

if MAX_SAMPLES:
    cmd.extend(["--max_samples", str(MAX_SAMPLES)])

print(f"Running QA BT {lang}...")
subprocess.run(cmd, check=True)
print(f"✓ QA BT {lang} complete!")

## Step 4e: QA - Chinese (zh-CN)

In [ ]:
# QA BT - Chinese
lang = "zh-CN"

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/qa_entity.py",
    "--mode", "bt",
    "--lang", lang,
    "--input_path", QG_OUTPUT,
    "--bt_path", ORIGINAL_DATASET_PATH,
    "--output_path", f"{QA_BT_DIR}/bt-{lang}.jsonl"
]

if MAX_SAMPLES:
    cmd.extend(["--max_samples", str(MAX_SAMPLES)])

print(f"Running QA BT {lang}...")
subprocess.run(cmd, check=True)
print(f"✓ QA BT {lang} complete!")

## Step 5: Mapping

Combine source and BT answers with entity-level breakdown.

In [ ]:
# Mapping

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/mapping.py",
    "--qa_source_path", QA_SOURCE_OUTPUT,
    "--qa_bt_dir", QA_BT_DIR,
    "--bt_original_path", ORIGINAL_DATASET_PATH,
    "--output_path", MAPPED_OUTPUT
]

print("Running Mapping...")
subprocess.run(cmd, check=True)
print("✓ Mapping complete!")

## Step 6: String Comparison Evaluation

Calculate F1, EM with entity-type breakdown.

In [ ]:
# String Comparison

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/string_comparison.py",
    "--input_path", MAPPED_OUTPUT,
    "--output_dir", STRING_COMP_OUTPUT
]

print("Running String Comparison...")
subprocess.run(cmd, check=True)
print("✓ String Comparison complete!")

## Step 7: SBERT Evaluation

Calculate semantic similarity with entity-type breakdown.

In [ ]:
# SBERT

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/sbert.py",
    "--input_path", MAPPED_OUTPUT,
    "--output_dir", SBERT_OUTPUT
]

print("Running SBERT Evaluation...")
subprocess.run(cmd, check=True)
print("✓ SBERT complete!")

## Verification

Check that all outputs were created.

In [ ]:
# Verification

expected_files = [
    NER_OUTPUT,
    QG_OUTPUT,
    QA_SOURCE_OUTPUT,
    MAPPED_OUTPUT,
] + [f"{QA_BT_DIR}/bt-{lang}.jsonl" for lang in LANGUAGES]

print("Checking output files...")
print("=" * 50)

all_ok = True
for f in expected_files:
    exists = os.path.exists(f)
    status = "✓" if exists else "✗"
    print(f"{status} {os.path.basename(f)}")
    if not exists:
        all_ok = False

print("=" * 50)
if all_ok:
    print("All files created successfully!")
else:
    print("WARNING: Some files are missing!")

## Summary

Pipeline complete! Check the evaluation outputs for entity-type breakdown analysis.

In [ ]:
print("\n" + "="*60)
print("NER-ENHANCED ASKQE PIPELINE COMPLETE")
print("="*60)
print(f"\nOutputs saved to: {EXTENSION_DIR}")
print(f"\nEvaluation results:")
print(f"  - String Comparison: {STRING_COMP_OUTPUT}")
print(f"  - SBERT: {SBERT_OUTPUT}")
print("\nCheck the entity-type breakdown in the evaluation output files!")